In [78]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# 使用本地模型（无需下载）
model_name = "/Users/tong/Documents/models/Qwen/Qwen2.5-1.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [79]:
# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=50,
    do_sample=False,
)


In [80]:
# Generate text
prompt = ("Write an email apologizing to Sarah for the late reply.")
output = generator(prompt)
print(output[0]["generated_text"])


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Subject: Apologies for the Delayed Response

Dear Sarah,

I hope this message finds you well.

I wanted to take a moment to express my sincerest apologies for the delay in responding to your recent inquiry regarding our project timeline. I understand how


In [81]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [82]:
prompt = "The capital of France is"

In [83]:
# Tokenize the input prompt（确保与 model 同设备，避免 MPS/CPU 不匹配）
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
input_ids


tensor([[ 785, 6722,  315, 9625,  374]], device='mps:0')

In [84]:
model_output = model.model(input_ids)
model_output[0].shape

torch.Size([1, 5, 1536])

In [85]:
lm_head_output = model.lm_head(model_output[0])
lm_head_output.shape

torch.Size([1, 5, 151936])

In [86]:
token_id = lm_head_output[0,-1].argmax(dim=-1)
token_id
tokenizer.decode(token_id)

' Paris'